# Bundle Clustering

## 1. Importing / Installing Packages

In [1]:
from __future__ import annotations

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from matplotlib import colors  # ✅ needed import

%matplotlib inline
%config InlineBackend.figure_format = 'svg'

import folium

from sklearn.cluster import DBSCAN
from scipy.spatial import ConvexHull

from bayes_opt import BayesianOptimization

from typing import Dict, Optional, Sequence, Mapping, Any

import geopandas as gpd
from shapely.wkt import loads as wkt_loads

from src.utils import read_csv_with_mapper

## 3. Data Import

### 3.1 Reading Filtered Header and midpoints

In [2]:
# Reading WellHeader excel file to dataframe
df_raw_wellheader = pd.read_csv(
    # r"C:\Users\apoorva.saxena\Desktop\Projects_AP\01. Ring Energy\Well Header\Header_Filtered.csv"
    r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\For Matt\Well Header\Header_Filtered.csv"
        ,usecols=["uwi", "well_name", "operator", "rsv_cat", "bench"] ,dtype={"uwi":str})

df_raw_wellheader = df_raw_wellheader[df_raw_wellheader["rsv_cat"]!="03PUD"].reset_index(drop=True).copy()

In [3]:
df_raw_wellheader

,uwi,well_name,operator,rsv_cat,bench
0,42003002890200,UNIVERSITY BLOCK 9 4AB,Exxon Mobil,02PDNP,MISSISSIPPIAN
1,42003029150100,UNIVERSITY BLOCK 9 6AO,Exxon Mobil,02PDNP,SUB-WOODFORD
2,42003029190100,UNIVERSITY BLOCK 9 7AO,Exxon Mobil,02PDNP,SUB-WOODFORD
3,42003029190200,UNIVERSITY BLOCK 9 7AO,Exxon Mobil,02PDNP,SUB-WOODFORD
4,42003029720300,UNIVERSITY BLOCK 9 3AT,Exxon Mobil,01PDP,SUB-WOODFORD
...,...,...,...,...,...
1875,42501375990000,RED RAIDER 663 A 6H,Ring Energy,01PDP,SAN ANDRES
1876,42501376000000,RED RAIDER 663 B 7H,Ring Energy,01PDP,SAN ANDRES
1877,42501376010000,MF 732-733 1H,Amtex Energy,01PDP,SAN ANDRES
1878,42501376030000,RED RAIDER 663 C 8H,Ring Energy,01PDP,SAN ANDRES


### 3.2 Reading Mid-Points data frame

In [4]:
df_midpoints = pd.read_csv(
    # r"C:\Users\apoorva.saxena\Desktop\Projects_AP\01. Ring Energy\Directional Surveys\Lateral_Midpoints.csv",
    r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\For Matt\Directional Surveys\Lateral_Midpoints.csv",
                           dtype={"uwi":str})

In [5]:
df_midpoints

,uwi,heel_lat,heel_lon,toe_lat,toe_lon,mid_Lat,mid_Lon
0,30025410040100,33.128186,-103.063231,33.139705,-103.063334,33.133946,-103.063283
1,30025421210000,33.113373,-103.069194,33.125204,-103.069167,33.119289,-103.069181
2,30025426220000,33.113377,-103.072581,33.125239,-103.073067,33.119308,-103.072824
3,30025428730000,33.113636,-103.063432,33.125201,-103.063393,33.119418,-103.063413
4,30025435920100,33.097298,-103.074979,33.111496,-103.075177,33.104397,-103.075078
...,...,...,...,...,...,...,...
2010,42501376070000,33.095593,-102.908611,33.124152,-102.907813,33.109872,-102.908212
2011,42501376080000,33.095416,-102.910903,33.124376,-102.910116,33.109896,-102.910510
2012,42501376090000,33.074192,-102.904032,33.095278,-102.903988,33.084735,-102.904010
2013,42501376100000,33.095262,-102.913369,33.109847,-102.913430,33.102554,-102.913399


### 3.3 Reading directional survey

In [6]:
def surveys_to_linestring_df(
    survey_df: pd.DataFrame,
    header_df: Optional[pd.DataFrame] = None,
    header_cols: Optional[Sequence[str]] = None,
    *,
    uwi_col: str = "uwi",
    lat_col: str = "latitude",
    lon_col: str = "longitude",
    md_col: str = "md",
    include_z: bool = False,
    z_col: str = "tvd",
    linestring_col: str = "geom_wkt",
    drop_single_point_wells: bool = True,
) -> pd.DataFrame:
    """
    Convert directional survey points into one WKT LINESTRING per well,
    optionally joined to a header table.

    The resulting DataFrame is ready to export to CSV / Parquet and can be
    read directly by Spotfire or QGIS (using `linestring_col` as the
    geometry column, CRS = EPSG:4326).

    Parameters
    ----------
    survey_df :
        Directional survey DataFrame with one row per survey station.
        Must contain at least:
        - uwi_col (default 'uwi')
        - lat_col (default 'latitude')  in decimal degrees
        - lon_col (default 'longitude') in decimal degrees
        - md_col  (default 'md') for sorting along the well path
        If `include_z=True`, also needs `z_col` (default 'tvd').

    header_df :
        Optional header DataFrame with one row per well (or several rows
        that can be deduplicated by `uwi_col`). This is joined after
        the LINESTRING is built.

    header_cols :
        Which columns from `header_df` to keep. If None, all columns in
        `header_df` are kept.

    uwi_col, lat_col, lon_col, md_col, z_col :
        Column names in `survey_df` / `header_df`. Adjust if your naming
        is different.

    include_z :
        If True, create a 3D LINESTRING Z (lon lat z). Otherwise create a
        2D LINESTRING (lon lat).

    linestring_col :
        Name of the output column containing WKT text.

    drop_single_point_wells :
        If True, wells with fewer than 2 valid survey points are dropped.
        If False, they are kept; in that case a POINT WKT is returned
        instead of a LINESTRING.

    Returns
    -------
    out_df :
        DataFrame with one row per well, containing:
        - uwi_col
        - linestring_col : WKT string (LINESTRING or LINESTRING Z)
        - any requested header columns (if header_df is provided)
    """
    # Work on a copy to avoid modifying the original
    df = survey_df.copy()

    # Ensure ordered along the well path
    df = df.sort_values([uwi_col, md_col])

    # Drop rows missing coordinates
    coord_cols = [lat_col, lon_col]
    if include_z:
        coord_cols.append(z_col)

    df = df.dropna(subset=coord_cols)

    records = []

    for uwi, grp in df.groupby(uwi_col, sort=False):
        if grp.empty:
            continue

        if len(grp) < 2 and drop_single_point_wells:
            # Not enough points to make a line – skip
            continue

        if include_z:
            coords = grp[[lon_col, lat_col, z_col]].to_numpy()
            coord_str = ", ".join(f"{x} {y} {z}" for x, y, z in coords)
            wkt = f"LINESTRING Z ({coord_str})"
        else:
            coords = grp[[lon_col, lat_col]].to_numpy()
            coord_str = ", ".join(f"{x} {y}" for x, y in coords)
            wkt = f"LINESTRING ({coord_str})"

        records.append({uwi_col: uwi, linestring_col: wkt})

    out_df = pd.DataFrame.from_records(records)

    # Attach header columns if provided
    if header_df is not None and not header_df.empty:
        if header_cols is None:
            header_use = header_df.copy()
        else:
            header_use = header_df[[uwi_col, *header_cols]].copy()

        header_use = header_use.drop_duplicates(subset=[uwi_col])
        out_df = out_df.merge(header_use, on=uwi_col, how="left")

    return out_df

In [7]:
df_directional_survey = pd.read_csv(
    # r"C:\Users\apoorva.saxena\Desktop\Projects_AP\01. Ring Energy\Directional Surveys\Directional_Survey.csv",
    r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\For Matt\Directional Surveys\Directional_Survey.csv",
    dtype={"uwi":str})

# Filter directional survey data to include only wells present in the raw well header data
df_directional_survey = df_directional_survey[df_directional_survey["uwi"].isin(df_raw_wellheader["uwi"].unique())].reset_index(drop=True).copy()

# Convert survey points to linestrings
df_survey_linestring = surveys_to_linestring_df(survey_df=df_directional_survey, header_df=df_raw_wellheader)

In [8]:
df_survey_linestring

,uwi,geom_wkt,well_name,operator,rsv_cat,bench
0,30025410040100,"LINESTRING (-103.062995337 33.126830884, -103....",BROKEN SPOKE 2 STATE #001H,Burk Royalty,01PDP,SAN ANDRES
1,30025421210000,"LINESTRING (-103.069252768 33.112247194, -103....",DOG BAR 11 FEE #002H,Burk Royalty,01PDP,SAN ANDRES
2,30025426220000,"LINESTRING (-103.072580533 33.112225034, -103....",DOG BAR 11 FEE #003H,Burk Royalty,01PDP,SAN ANDRES
3,30025428730000,"LINESTRING (-103.063123047 33.112295036, -103....",DOG BAR 11 FEE #001H,Burk Royalty,01PDP,SAN ANDRES
4,30025435920100,"LINESTRING (-103.075096391 33.095841751, -103....",PINKMAN FEE #004H,Steward Energy II,02PA,SAN ANDRES
...,...,...,...,...,...,...
1875,42501375990000,"LINESTRING (-103.002091078 33.08165402, -103.0...",RED RAIDER 663 A 6H,Ring Energy,01PDP,SAN ANDRES
1876,42501376000000,"LINESTRING (-103.006275157 33.082538911, -103....",RED RAIDER 663 B 7H,Ring Energy,01PDP,SAN ANDRES
1877,42501376010000,"LINESTRING (-102.90883919 33.04218437, -102.90...",MF 732-733 1H,Amtex Energy,01PDP,SAN ANDRES
1878,42501376030000,"LINESTRING (-103.006373059 33.082534816, -103....",RED RAIDER 663 C 8H,Ring Energy,01PDP,SAN ANDRES


### 3.4 Reading wps and average spacing data

In [11]:
df_wps = read_csv_with_mapper(
    r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\Python\Parent_Child_Spacing\data\wps_summary.csv",
    col_map={"well_i": "uwi"}, dtype_map={"uwi": str})

df_avg_spacing = read_csv_with_mapper(
    r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\Python\Parent_Child_Spacing\data\avg_spacing.csv",
    col_map={"well_i": "uwi"}, dtype_map={"uwi": str}
)

In [12]:
df_avg_spacing

,uwi,avg_hz_spacing_ft,avg_vt_spacing_ft,neighbors_considered,adjacency_coverage_i_pct
0,30025410040100,933.507723,23.612500,1,1.000000
1,30025421210000,1107.525915,26.044500,1,0.999413
2,30025426220000,1107.539330,26.044500,1,0.986746
3,30025428730000,NaN,NaN,0,NaN
4,30025437350100,NaN,NaN,0,NaN
...,...,...,...,...,...
1859,42501376070000,840.355942,26.424377,3,1.000000
1860,42501376080000,753.157628,19.092379,3,1.000000
1861,42501376090000,906.587087,68.695000,1,0.947186
1862,42501376100000,766.016578,23.225298,2,1.000000


In [15]:
df_wps

,uwi,wps_cardinal,wps_iframe,wps_corridor,anisotropy_ratio,anisotropy_delta,azimuth_deg,mid_x,mid_y
0,30025410040100,3,3,2,1.0,0.0,358.510488,2.233137e+06,1.203348e+07
1,30025421210000,3,3,3,1.0,0.0,359.055498,2.231430e+06,1.202812e+07
2,30025426220000,3,3,3,1.0,0.0,356.969534,2.230315e+06,1.202810e+07
3,30025428730000,3,3,3,1.0,0.0,359.103991,2.233195e+06,1.202820e+07
4,30025435920100,1,1,1,1.0,0.0,358.276644,2.229724e+06,1.202267e+07
...,...,...,...,...,...,...,...,...,...
2010,42501376070000,12,12,12,1.0,0.0,0.204329,2.280778e+06,1.202564e+07
2011,42501376080000,10,10,10,1.0,0.0,0.168816,2.280075e+06,1.202563e+07
2012,42501376090000,5,5,5,1.0,0.0,358.957836,2.282248e+06,1.201652e+07
2013,42501376100000,5,5,5,1.0,0.0,358.657432,2.279243e+06,1.202294e+07


## 4. Data Preprocessing

### 4.1 Filtering midpoints to only include those present in the wellheader

In [13]:
df_midpoints_filter = df_midpoints[df_midpoints["uwi"].isin(df_raw_wellheader["uwi"].unique())].reset_index(drop=True).copy()

## 5. Feature Engineering

In [14]:
# df_midpoints_filter.plot(kind='scatter',x='mid_Lat',y='mid_Lon',figsize=(19,8))

## 6. DBSCAN

### 6.1 Defining Functions

In [11]:
def haversine_distance(lon1, lat1, lon2, lat2,**kwargs):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    All args must be of equal length.

    """
    # convert degrees to radians
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))

    # earth radius in km (match the DBSCAN constant)
    earth_radius_km = 6371.0088
    distance_km = earth_radius_km * c

    feet_per_km = 3280.84
    distance_ft = distance_km * feet_per_km

    return distance_ft

def dbscan_cluster(latitudes, longitudes, epsilon, min_samples, **kwargs):
    '''
    Function to perform DBSCAN clustering for given parameters.

    epsilon is provided in **feet**, converted to radians for the
    sklearn haversine metric.
    '''
    # constants
    kms_per_radian = 6371.0088        # km per radian on Earth's surface
    feet_per_km = 3280.84

    # convert epsilon from feet -> km -> radians
    epsilon_km = epsilon / feet_per_km
    epsilon_rad = epsilon_km / kms_per_radian

    dbscan = DBSCAN(
        eps=epsilon_rad,
        min_samples=min_samples,
        algorithm='ball_tree',
        metric='haversine',
        **kwargs
    )

    dbscan.fit(
        np.radians([x for x in zip(latitudes, longitudes)])
    )

    return pd.Series(dbscan.labels_)

def vertex_centroid_distance(latitudes,longitudes,**kwargs):
    '''
    Function to calculate the average distance from the vertices of a convex hull
    (derived from latitude x longitude pairs) to the centroid of said convex hull.
    
    Centroid is taken to be the unweighted average of all co-ordinate pairs.
    
    '''
    
    # co-ordinates of centre
    # take a simple average
    centre_long = longitudes.mean()
    centre_lats = latitudes.mean()
    
    # collapse two points into line
    if len(latitudes) < 3:
        distances = haversine_distance(
            longitudes,
            latitudes,
            centre_long,
            centre_lats,
            **kwargs).mean()
    
    else:
        # convex hull
        convex_hull = ConvexHull([x for x in zip(latitudes,longitudes)],**kwargs)

        # now get co-ordinates of vertices
        vertex_longs = longitudes.iloc[convex_hull.vertices]
        vertex_lats = latitudes.iloc[convex_hull.vertices]

        # now get
        distances = haversine_distance(
            vertex_longs,
            vertex_lats,
            centre_long,
            centre_lats,
            **kwargs).mean()

    # return average distance
    return distances.mean() if not np.isnan(distances) else 0.0

def calculate_average_values_in_disctionary(dictionary: Dict[int, float]) -> Optional[float]:
    """
    Return the arithmetic mean of the values in `dictionary` (positive).

    Parameters
    ----------
    dictionary : dict[int, float]
        Mapping from cluster_id -> average distance in feet.

    Returns
    -------
    float or None
        Positive average of the values, or None if the dict is empty.
    """
    if not dictionary:
        return None
    return float(np.mean(list(dictionary.values())))

def black_box_function(epsilon: float, min_samples: float) -> float:
    """
    Objective for Bayesian Optimization.

    Returns
    -------
    target : float
        Negative average cluster radius in feet
        (maximizing target == minimizing avg radius).
    """
    # 1) Prepare data
    df = df_midpoints_filter[['mid_Lat', 'mid_Lon']].copy()
    df.drop_duplicates(inplace=True)

    # 2) DBSCAN with epsilon in **feet**
    df['cluster'] = dbscan_cluster(
        latitudes=df['mid_Lat'],
        longitudes=df['mid_Lon'],
        epsilon=epsilon,
        min_samples=int(min_samples)
    )

    # 3) Per-cluster distances (feet)
    vertex_dist: Dict[int, float] = {}
    for cluster_id in df['cluster'].unique():
        df_cluster = df[df['cluster'] == cluster_id][['mid_Lat', 'mid_Lon']].copy()

        vertex_dist[cluster_id] = vertex_centroid_distance(
            latitudes=df_cluster['mid_Lat'],
            longitudes=df_cluster['mid_Lon']
        )

    # 4) Average radius across clusters (feet)
    avg_radius_ft = calculate_average_values_in_disctionary(vertex_dist)

    if avg_radius_ft is None or not np.isfinite(avg_radius_ft):
        # Neutral fallback if something degenerate happens
        return 0.0

    target = -avg_radius_ft
    return target

def build_bo_results_df(optimizer) -> pd.DataFrame:
    """
    Flatten optimizer.res into a DataFrame with
    iter, epsilon, min_samples, target, avg_radius_ft.
    """
    res = pd.DataFrame(optimizer.res)          # columns: ['target', 'params']
    params = res['params'].apply(pd.Series)    # split params into columns

    df = pd.concat([params, res['target']], axis=1)
    df.insert(0, 'iter', range(1, len(df) + 1))
    df['avg_radius_ft'] = -df['target']        # convert back to positive

    # nice column order
    return df[['iter', 'target', 'avg_radius_ft', 'epsilon', 'min_samples']]

def plot_clusters(df: pd.DataFrame, eps: float, min_samp: int):
    df_result = df.copy()

    df_result['cluster'] = dbscan_cluster(
        latitudes=df_result['mid_Lat'],
        longitudes=df_result['mid_Lon'],
        epsilon=eps,
        min_samples=min_samp
    )

    m = folium.Map(
        location=[df_result['mid_Lat'].mean(), df_result['mid_Lon'].mean()],
        tiles="OpenStreetMap",
        zoom_start=11
    )

    # Create a colormap for the unique cluster labels
    cmap = plt.get_cmap('hsv', len(df_result['cluster'].unique()))

    # Create a color dictionary for each unique cluster
    color_map = {cluster: colors.rgb2hex(cmap(i)) 
                 for i, cluster in enumerate(df_result['cluster'].unique())}

    # Add a circle marker for each point
    for _, row in df_result.iterrows():
        folium.CircleMarker(
            location=[row['mid_Lat'], row['mid_Lon']],
            radius=5,
            color="white",
            fill=True,
            popup=f"{row['well_name']}, Cluster: {row['cluster']}",
            fill_color=color_map[row['cluster']],
            fill_opacity=1
        ).add_to(m)

    return m, df_result

### 6.2 Running Bayesian Optimization

In [12]:
# Bounded region of parameter space
pbounds = {'epsilon': (800, 2000), 'min_samples': (1, 2)}

optimizer = BayesianOptimization(
    f=black_box_function,
    pbounds=pbounds,
    random_state=0,
    allow_duplicate_points=True
)

optimizer.maximize()

results_df = build_bo_results_df(optimizer)

results_df.sort_values('epsilon', ascending=True)

|   iter    |  target   |  epsilon  | min_sa... |
-------------------------------------------------
| 1         | -9.555e+0 | 1.459e+03 | 1.715     |
| 2         | -1.063e+0 | 1.523e+03 | 1.545     |
| 3         | -8.219e+0 | 1.308e+03 | 1.646     |
| 4         | -8.54e+03 | 1.325e+03 | 1.892     |
| 5         | -1.323e+0 | 1.956e+03 | 1.383     |
| 6         | -6.053e+0 | 1.14e+03  | 1.0       |
| 7         | -6.053e+0 | 1.141e+03 | 1.907     |
| 8         | -2.423e+0 | 832.5     | 2.0       |
| 9         | -6.826e+0 | 1.196e+03 | 1.0       |
| 10        | -3.065e+0 | 1.75e+03  | 2.0       |
| 11        | -2.509e+0 | 1.036e+03 | 2.0       |
| 12        | -3.298e+0 | 2e+03     | 2.0       |
| 13        | -1.277e+0 | 1.893e+03 | 1.079     |
| 14        | -9.16e+03 | 1.395e+03 | 1.0       |
| 15        | -1.102e+0 | 1.606e+03 | 1.0       |
| 16        | -2.723e+0 | 1.248e+03 | 2.0       |
| 17        | -6.697e+0 | 1.169e+03 | 1.094     |
| 18        | -4.976e+0 | 933.4     | 1.0       |


,iter,target,avg_radius_ft,epsilon,min_samples
7,8,-24232.164869,24232.164869,832.487798,2.000000
21,22,-24231.180484,24231.180484,896.500840,2.000000
17,18,-4976.108287,4976.108287,933.422104,1.000000
27,28,-25073.202654,25073.202654,949.926977,2.000000
18,19,-5181.868530,5181.868530,966.953786,1.955162
29,30,-5297.924646,5297.924646,979.505210,1.563849
10,11,-25089.479975,25089.479975,1036.353246,2.000000
22,23,-25444.934410,25444.934410,1102.235557,2.000000
5,6,-6053.122051,6053.122051,1140.245922,1.000000
6,7,-6053.122051,6053.122051,1140.796720,1.907377


### 6.3 Getting and Plotting Final DB Cluster

In [13]:
m, df_with_clusters = plot_clusters(df=df_midpoints_filter.join(df_raw_wellheader.set_index('uwi'),on='uwi', how='left'), 
                                    eps=1325, min_samp=1)

In [14]:
df_with_clusters[df_with_clusters.duplicated(subset=["cluster"], keep=False)].sort_values("cluster")

,uwi,heel_lat,heel_lon,toe_lat,toe_lon,mid_Lat,mid_Lon,well_name,operator,rsv_cat,bench,cluster
0,30025410040100,33.128186,-103.063231,33.139705,-103.063334,33.133946,-103.063283,BROKEN SPOKE 2 STATE #001H,Burk Royalty,01PDP,SAN ANDRES,0
33,30025503690000,33.126181,-103.066415,33.140744,-103.066344,33.133463,-103.066380,BROKEN SPOKE 2 STATE #002H,Burk Royalty,01PDP,SAN ANDRES,0
1,30025421210000,33.113373,-103.069194,33.125204,-103.069167,33.119289,-103.069181,DOG BAR 11 FEE #002H,Burk Royalty,01PDP,SAN ANDRES,1
2,30025426220000,33.113377,-103.072581,33.125239,-103.073067,33.119308,-103.072824,DOG BAR 11 FEE #003H,Burk Royalty,01PDP,SAN ANDRES,1
35,30025507280000,33.133624,-103.080061,33.155285,-103.079888,33.144455,-103.079974,HEISENBERG STATE COM #002H,Burk Royalty,01PDP,SAN ANDRES,6
...,...,...,...,...,...,...,...,...,...,...,...,...
1840,42501374780000,33.212939,-103.015477,33.226269,-103.014741,33.219604,-103.015109,FREDDY FALCON 360 3H,Ring Energy,01PDP,SAN ANDRES,908
1847,42501374990000,33.117276,-102.877854,33.135161,-102.877990,33.126218,-102.877922,CASSIDY 575-544 E 4XH,Riley Permian,01PDP,SAN ANDRES,910
1846,42501374980000,33.117151,-102.880213,33.135178,-102.880229,33.126165,-102.880221,CASSIDY 575-544 5XH,Riley Permian,01PDP,SAN ANDRES,910
1869,42501375570000,33.065690,-102.876076,33.051661,-102.876058,33.058675,-102.876067,KNOTTED ROPE E 703 3H,Riley Permian,01PDP,SAN ANDRES,915


In [15]:
# Merge survey linestrings with cluster assignments
df_survey_linestring_cluster = df_survey_linestring.merge(df_with_clusters[["uwi", "cluster"]], on="uwi", how="left").reset_index(drop=True).copy()

gdf = gpd.GeoDataFrame(
    df_survey_linestring_cluster.drop(columns=['geom_wkt']),
    geometry=df_survey_linestring_cluster['geom_wkt'].apply(wkt_loads),
    crs="EPSG:4326"
)

# Option A: shapefile (folder with .shp/.dbf/.shx/.prj)
gdf.to_file(r"C:\Users\apoorva.saxena\Desktop\Projects_AP\01. Ring Energy\shapefiles\well_lines.shp")

# Option B: GeoPackage (single file)
# gdf.to_file("well_lines.gpkg", layer="well_lines", driver="GPKG")

In [16]:
df_survey_linestring_cluster.groupby("cluster")["uwi"].count()

cluster
0      2
1      2
2      1
3      1
4      1
      ..
915    2
916    1
917    1
918    1
919    1
Name: uwi, Length: 920, dtype: int64